# Neuronal Coactivity by Vocal Category — Analyses

This notebook quantifies how coordinated a PAG (or other region) neural population is during one class of ultrasonic vocalizations (USVs) versus another, using the pairwise spike-count correlation, population-vector cosine similarity, and population-vector Pearson correlation computed by `usv_playpen.analyses.neuronal_coactivity_engine`.

The workflow is: pooled trial-count bootstrap to a matched N, a chained circular-shuffle null per group, and a direct group-A-vs-group-B label-permutation test — reported as summary tables, per-metric null-distribution plots, a per-session breakdown, and a cross-animal slope plot.

The cells are organised as *(1) imports*, *(2) parameters — every tweakable knob*, *(3) setup & data load*, then *(4) per-section compute + plot*. Each compute/plot cell consumes the **Imports**, **Parameters**, and **Setup & load** cells; re-run those first.

In [ ]:
### Imports

import pathlib

import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as st

from usv_playpen.os_utils import configure_path
from usv_playpen.visualizations.plot_style import apply_plot_style
import usv_playpen.analyses.neuronal_coactivity_engine as engine

apply_plot_style()

## Parameters

Every knob a user might tweak for a run lives in the single cell below — trial segmentation, unit filters, data paths, the animal→sessions map, the coactivity hyperparameters, and the per-group plotting colours. Edit here; nothing downstream redefines these. Paths are written `/mnt/falkner/...` and wrapped in `configure_path()` so they resolve on macOS (`/Volumes/falkner`) too.

In [ ]:
### Parameters — every user-tweakable knob lives here

# Segmentation configuration
CATEGORY_COLUMN = "qlvm_supercategory"
GROUP_A_IDS   = [1]
GROUP_A_LABEL = "complex"
GROUP_B_IDS   = [7]
GROUP_B_LABEL = "simple"

# Unit-filter configuration (3 criteria: cluster_group + somatic + brain area)
CATALOG_PATH       = pathlib.Path(configure_path("/mnt/falkner/Bartul/EPHYS/unit_catalog.csv"))
UNIT_BRAIN_AREAS   = {"PAG"}
UNIT_REQUIRE_SOMATIC = True
UNIT_CLUSTER_GROUP = "good"

# Animal -> sessions mapping
# Sessions can span multiple recording days. Kilosort is run per day,
# so cluster IDs aren't stable across days; the loader picks the
# single-day block with the MOST units passing the catalog filters,
# breaking ties by session count. Days where the probe wasn't in the
# requested brain area (post-hoc histology) are automatically skipped.
DATA_ROOT = pathlib.Path(configure_path("/mnt/falkner/Bartul/Data"))

ANIMALS_TO_SESSIONS: dict[str, list[str]] = {
    "158112_0": [
        "20241107_135544", "20241107_143336", "20241107_154636",
        "20241107_172830",
    ],
    "158114_2": [
        "20241115_165038", "20241115_172055", "20241115_185527",
        "20241115_192700",
    ],
    "164335_0": [
        "20250909_151929", "20250909_154745",
        "20250911_154739", "20250911_162235", "20250911_165912",
        "20250911_182910", "20250911_190415", "20250911_194127",
        "20250912_125331", "20250912_150922", "20250912_155546",
        "20250912_174643", "20250912_181920", "20250912_185338",
        "20250913_143737", "20250913_150744", "20250913_153700",
        "20250913_173732", "20250913_180808", "20250913_183548",
    ],
    "181316_0": [
        "20250919_152924", "20250919_155842", "20250919_163351",
        "20250923_162249", "20250923_171803", "20250923_174642",
        "20250923_184326", "20250923_200538", "20250923_203320",
    ],
    "178621_2": [
        "20250927_142335", "20250927_145144", "20250927_151825",
        "20250927_172337", "20250927_175128", "20250927_181936",
        "20250928_172408", "20250928_175135", "20250928_182348",
        "20250928_192858", "20250928_195647", "20250928_202420",
    ],
    "181321_1": [
        "20251003_140135", "20251003_143416", "20251003_150452",
        "20251003_161312", "20251003_164534", "20251003_171558",
        "20251004_154544", "20251004_162927", "20251004_170028",
        "20251004_180923", "20251004_183841", "20251004_190735",
    ],
    "181322_2": [
        "20251011_140347", "20251011_145914", "20251011_153450",
        "20251011_174924", "20251011_182556", "20251011_190000",
        "20251012_150953", "20251012_154216", "20251012_161241",
        "20251012_171752", "20251012_174634", "20251012_181640",
    ],
}

CHOSEN_ANIMAL = "178621_2"

# Coactivity hyperparameters
SEED = 0   # base RNG seed; each stochastic routine draws an independent, reproducible stream as SEED + offset
USV_BOOTSTRAP_NUM = 300
N_BOOT_ITERATIONS = 1000
N_SHUFFLES = 1000
N_PERMUTATIONS = 1000
WINDOW_S = 0.030
PER_SESSION_N_SHUFFLES = 500   # smaller than the chained null since we run per session

# Group plotting colours (hex). Group A uses crimson, group B uses dodgerblue
# by default — change here to retune. Labels come from the segmentation config above.
GROUP_A_COLOR = "#DC143C"
GROUP_B_COLOR = "#1E90FF"
NULL_COLOR    = "#808080"
THRESHOLD_COLOR = "#000000"

## Setup & load data

Loads the chosen animal's data through `engine.load_unit_catalog` + `engine.load_animal_sessions` — the unit-catalog read, the three-criteria unit filter (`cluster_group` + `somatic` + `brain_area`), the single-best-day population selection (Kilosort is per-day, so units aren't comparable across days), and the per-session category split now all live in `neuronal_coactivity_engine`. You should not need to edit this cell; change inputs in **Parameters** above.

In [ ]:
### Setup & load data

catalog = engine.load_unit_catalog(CATALOG_PATH)

print(f"Trial split:  `{CATEGORY_COLUMN}`")
print(f"  group A ({GROUP_A_LABEL}) = IDs {GROUP_A_IDS}")
print(f"  group B ({GROUP_B_LABEL}) = IDs {GROUP_B_IDS}")
print(f"Unit filter:  cluster_group='{UNIT_CLUSTER_GROUP}'  "
      f"somatic={UNIT_REQUIRE_SOMATIC}  brain_area in {sorted(UNIT_BRAIN_AREAS) or 'ANY'}")
print(f"Chosen animal: {CHOSEN_ANIMAL}")

sessions_data = engine.load_animal_sessions(
    CHOSEN_ANIMAL, ANIMALS_TO_SESSIONS[CHOSEN_ANIMAL],
    data_root=DATA_ROOT, catalog=catalog,
    category_column=CATEGORY_COLUMN, group_a_ids=GROUP_A_IDS, group_b_ids=GROUP_B_IDS,
    cluster_group=UNIT_CLUSTER_GROUP, require_somatic=UNIT_REQUIRE_SOMATIC,
    brain_areas=UNIT_BRAIN_AREAS,
)
n_common = len(next(iter(sessions_data))['neural_data']) if sessions_data else 0
print(f"Loaded {len(sessions_data)} sessions for {CHOSEN_ANIMAL}; common filtered units = {n_common}")

## Acoustic confound check

The fixed 30 ms window equalises call **duration**, but the two categories could still differ in the **acoustics** of that window — chiefly loudness and pitch — which might drive population activity independently of call identity. For every call this section reads the loudest-channel (`peak_amp_ch`) waveform snippet `[onset, onset + WINDOW_S)` from the session's `int16` audio and computes four features via `engine.extract_snippet_acoustics`: absolute **RMS amplitude** (from the raw waveform) plus **mean / peak frequency** and **frequency bandwidth** (energy-weighted, from the snippet's linear-power spectrogram — see the note below). It then compares the pooled `complex`-vs-`simple` distributions per feature (Mann–Whitney U + Cohen's *d*).

This is **diagnostic only** — it establishes *which* features differ between the groups (and by how much) so a control can be chosen afterwards. The headline coactivity metric `pop_corr` is mean-centred and already removes per-trial firing rate, so a future control mainly needs to guard against acoustic effects on coactivity *structure*.

**Note on the frequency features:** `mean_freq` / `freq_bandwidth` are computed as energy-weighted (linear-power) spectral centroid / spread — the textbook definitions — rather than on the cohort's `power_to_db(ref=max)` + min-max-normalised spectrogram. The cohort form relies on a per-USV region mask to suppress the noise floor; these maskless 30 ms snippets have none, so linear power gives a sharper, physically meaningful measure (`peak_freq` is identical either way). RMS is an absolute, raw-waveform measure.

In [ ]:
### Acoustic confound check: are the groups matched in amplitude & frequency?

# Features checked + their axis labels (kept here as this section's only knobs).
ACOUSTIC_FEATURES = ["rms", "mean_freq_hz", "peak_freq_hz", "freq_bandwidth_hz"]
ACOUSTIC_LABELS = {
    "rms":               "RMS amplitude (a.u.)",
    "mean_freq_hz":      "mean frequency (Hz)",
    "peak_freq_hz":      "peak frequency (Hz)",
    "freq_bandwidth_hz": "frequency bandwidth (Hz)",
}

# Pool the per-call features across the chosen animal's sessions, per group. The
# per-call extraction (loudest-channel 30 ms snippet -> RMS + spectral features)
# lives in engine.compute_group_acoustics / engine.extract_snippet_acoustics.
group_a_acoustics = {feature: [] for feature in ACOUSTIC_FEATURES}
group_b_acoustics = {feature: [] for feature in ACOUSTIC_FEATURES}
for sess in sessions_data:
    a_feats = engine.compute_group_acoustics(sess, "group_a_df", WINDOW_S)
    b_feats = engine.compute_group_acoustics(sess, "group_b_df", WINDOW_S)
    for feature in ACOUSTIC_FEATURES:
        group_a_acoustics[feature].append(a_feats[feature])
        group_b_acoustics[feature].append(b_feats[feature])
group_a_acoustics = {f: np.concatenate(v) if v else np.array([]) for f, v in group_a_acoustics.items()}
group_b_acoustics = {f: np.concatenate(v) if v else np.array([]) for f, v in group_b_acoustics.items()}

# Per-feature comparison: Mann-Whitney U (distribution shift) + Cohen's d (effect size).
n_a = int(np.isfinite(group_a_acoustics["rms"]).sum())
n_b = int(np.isfinite(group_b_acoustics["rms"]).sum())
print(
    f"Acoustic confound check — {CHOSEN_ANIMAL}: "
    f"{GROUP_A_LABEL} (n={n_a}) vs {GROUP_B_LABEL} (n={n_b})\n"
)
for feature in ACOUSTIC_FEATURES:
    a = group_a_acoustics[feature][np.isfinite(group_a_acoustics[feature])]
    b = group_b_acoustics[feature][np.isfinite(group_b_acoustics[feature])]
    if a.size < 2 or b.size < 2:
        print(f"  {feature:18s}: insufficient data ({a.size} / {b.size})")
        continue
    _, p_value = st.mannwhitneyu(a, b, alternative="two-sided")
    d = engine.cohens_d(a, b)
    print(
        f"  {feature:18s}: {GROUP_A_LABEL} median={np.median(a):>10.4g}  "
        f"{GROUP_B_LABEL} median={np.median(b):>10.4g}  "
        f"Cohen's d={d:+.3f}  Mann-Whitney p={p_value:.2e}"
    )

# Overlaid per-feature histograms (density), complex vs simple.
fig, axes = plt.subplots(1, len(ACOUSTIC_FEATURES), figsize=(4.0 * len(ACOUSTIC_FEATURES), 3.2))
for ax, feature in zip(axes, ACOUSTIC_FEATURES):
    a = group_a_acoustics[feature][np.isfinite(group_a_acoustics[feature])]
    b = group_b_acoustics[feature][np.isfinite(group_b_acoustics[feature])]
    pooled = np.concatenate([a, b]) if (a.size or b.size) else np.array([0.0, 1.0])
    lo, hi = float(pooled.min()), float(pooled.max())
    bins = np.linspace(lo, hi, 40) if hi > lo else 40
    ax.hist(a, bins=bins, density=True, alpha=0.5, color=GROUP_A_COLOR, label=GROUP_A_LABEL)
    ax.hist(b, bins=bins, density=True, alpha=0.5, color=GROUP_B_COLOR, label=GROUP_B_LABEL)
    ax.set_xlabel(ACOUSTIC_LABELS[feature], fontsize=9)
    ax.set_ylabel("density", fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle(
    f"30 ms acoustic features — {GROUP_A_LABEL} vs {GROUP_B_LABEL} ({CHOSEN_ANIMAL})",
    fontsize=11,
)
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

## Compute coactivity metrics: group A vs group B

In [ ]:
### Compute coactivity metrics: group A vs group B

# 1. Per-session observed count matrices, then trial-axis concat
group_a_mats, group_b_mats = [], []
for sess in sessions_data:
    a_mat = engine.extract_snippet_matrix(
        sess['group_a_df']['start'].to_numpy(), sess['neural_data'], WINDOW_S,
    )
    b_mat = engine.extract_snippet_matrix(
        sess['group_b_df']['start'].to_numpy(), sess['neural_data'], WINDOW_S,
    )
    group_a_mats.append(a_mat)
    group_b_mats.append(b_mat)

all_group_a_counts = np.hstack(group_a_mats)
all_group_b_counts = np.hstack(group_b_mats)
print(
    f"Aggregated: {all_group_a_counts.shape[1]} {GROUP_A_LABEL} | "
    f"{all_group_b_counts.shape[1]} {GROUP_B_LABEL} trials"
)

# 2. Pooled bootstrap to matched N
print(f"Bootstrapping pooled data to N={USV_BOOTSTRAP_NUM} ...")
all_group_a_boot = engine.bootstrap_coactivity_distribution(
    all_group_a_counts, USV_BOOTSTRAP_NUM, N_BOOT_ITERATIONS, seed=SEED,
)
all_group_b_boot = engine.bootstrap_coactivity_distribution(
    all_group_b_counts, USV_BOOTSTRAP_NUM, N_BOOT_ITERATIONS, seed=SEED + 1,
)

# 3. Chained circular-shuffle null per group
print(f"Generating matched-N chained null distributions ...")
session_group_a_onsets = engine.sample_onsets_across_sessions(
    sessions_data, 'group_a_df', USV_BOOTSTRAP_NUM, seed=SEED + 2,
)
session_group_b_onsets = engine.sample_onsets_across_sessions(
    sessions_data, 'group_b_df', USV_BOOTSTRAP_NUM, seed=SEED + 3,
)

group_a_chained_null = engine.perform_chained_circular_shuffle(
    session_onsets=session_group_a_onsets,
    session_neural_data=[sess['neural_data'] for sess in sessions_data],
    session_durations=[sess['total_duration'] for sess in sessions_data],
    window_s=WINDOW_S,
    n_shuffles=N_SHUFFLES,
    seed=SEED + 4,
)
group_b_chained_null = engine.perform_chained_circular_shuffle(
    session_onsets=session_group_b_onsets,
    session_neural_data=[sess['neural_data'] for sess in sessions_data],
    session_durations=[sess['total_duration'] for sess in sessions_data],
    window_s=WINDOW_S,
    n_shuffles=N_SHUFFLES,
    seed=SEED + 5,
)

# 4. Direct A-vs-B permutation test
# Tests `metric(A) - metric(B)` against a null built by shuffling
# group labels across the pooled trial set. Unlike the bootstrap-vs-
# shuffle test above (which only compares each group to its own
# random-timing null), this directly answers "do the two groups
# differ in coactivity?".
print(f"Running label permutation test (n_perm={N_PERMUTATIONS}) ...")
perm_results = engine.perform_label_permutation_test(
    counts_a=all_group_a_counts,
    counts_b=all_group_b_counts,
    n_permutations=N_PERMUTATIONS,
    seed=SEED + 6,
)

# 5. Per-session breakdown — does the effect hold session-by-session?
print("\nPer-session observed metrics (no bootstrap / no shuffle):")
print(
    f"  {'session':<22} {'n_a':>4} {'n_b':>4}"
    f"   {'r_sc Δ':>10}   {'sim Δ':>10}   {'pop Δ':>10}"
)
per_session_deltas = {m: [] for m in ('r_sc', 'similarity', 'pop_corr')}
for sess in sessions_data:
    a_mat = engine.extract_snippet_matrix(
        sess['group_a_df']['start'].to_numpy(), sess['neural_data'], WINDOW_S,
    )
    b_mat = engine.extract_snippet_matrix(
        sess['group_b_df']['start'].to_numpy(), sess['neural_data'], WINDOW_S,
    )
    if a_mat.shape[1] < 2 or b_mat.shape[1] < 2:
        print(
            f"  {sess['session_id']:<22} {a_mat.shape[1]:>4} {b_mat.shape[1]:>4}"
            "   (skipped, too few trials)"
        )
        continue
    m_a = engine.compute_coactivity_metrics(a_mat)
    m_b = engine.compute_coactivity_metrics(b_mat)
    deltas = {k: m_a[k] - m_b[k] for k in m_a}
    for k in per_session_deltas:
        per_session_deltas[k].append(deltas[k])
    print(
        f"  {sess['session_id']:<22} {a_mat.shape[1]:>4} {b_mat.shape[1]:>4}"
        f"   {deltas['r_sc']:>+10.4f}"
        f"   {deltas['similarity']:>+10.4f}"
        f"   {deltas['pop_corr']:>+10.4f}"
    )

print(f"\nSessions where {GROUP_A_LABEL} > {GROUP_B_LABEL}, per metric:")
for k, vals in per_session_deltas.items():
    n_pos = sum(1 for v in vals if v > 0)
    print(f"  {k:<12}: {n_pos}/{len(vals)} sessions")

# 6. Summary tables — boot-vs-chained-null p/Z via engine.bootstrap_vs_null_stats
print("\n" + "=" * 75)
print(f"{GROUP_A_LABEL.upper()} vs CHAINED NULL  (N={USV_BOOTSTRAP_NUM})")
print("=" * 75)
for m in ('r_sc', 'similarity', 'pop_corr'):
    obs, null, p, z = engine.bootstrap_vs_null_stats(all_group_a_boot[m], group_a_chained_null[m])
    print(f"  {m:<12}: boot={obs:+.4f} | null={null:+.4f} | p={p:.4f}  (Z={z:+.2f})")

print(f"\n{GROUP_B_LABEL.upper()} vs CHAINED NULL  (N={USV_BOOTSTRAP_NUM})")
print("=" * 75)
for m in ('r_sc', 'similarity', 'pop_corr'):
    obs, null, p, z = engine.bootstrap_vs_null_stats(all_group_b_boot[m], group_b_chained_null[m])
    print(f"  {m:<12}: boot={obs:+.4f} | null={null:+.4f} | p={p:.4f}  (Z={z:+.2f})")

print("\n" + "=" * 75)
print(
    f"DIRECT PERMUTATION: {GROUP_A_LABEL.upper()} vs {GROUP_B_LABEL.upper()}"
    f"  (n_perm={N_PERMUTATIONS})"
)
print("=" * 75)
for m in ('r_sc', 'similarity', 'pop_corr'):
    r = perm_results[m]
    print(
        f"  Δ {m:<12}: obs={r['observed_delta']:+.4f} | "
        f"null mean={r['null_mean']:+.4f} | "
        f"p({GROUP_A_LABEL}>{GROUP_B_LABEL})={r['p_a_gt_b']:.4f} | "
        f"p_two_tail={r['p_two_tailed']:.4f}  (Z={r['z_score']:+.2f})"
    )

## Plot per-metric chained-null distributions with observed bootstrap means

In [ ]:
### Plot per-metric chained-null distributions with observed bootstrap means

# 1. 99th-percentile thresholds from each chained null
a_rsc_99 = np.percentile(group_a_chained_null['r_sc'], 99)
a_sim_99 = np.percentile(group_a_chained_null['similarity'], 99)
a_pop_99 = np.percentile(group_a_chained_null['pop_corr'], 99)

b_rsc_99 = np.percentile(group_b_chained_null['r_sc'], 99)
b_sim_99 = np.percentile(group_b_chained_null['similarity'], 99)
b_pop_99 = np.percentile(group_b_chained_null['pop_corr'], 99)

# 2. Observed pooled-bootstrap means
a_val_rsc = np.mean(all_group_a_boot['r_sc'])
a_val_sim = np.mean(all_group_a_boot['similarity'])
a_val_pop = np.mean(all_group_a_boot['pop_corr'])

b_val_rsc = np.mean(all_group_b_boot['r_sc'])
b_val_sim = np.mean(all_group_b_boot['similarity'])
b_val_pop = np.mean(all_group_b_boot['pop_corr'])

# 3. 3 rows (metric) x 2 columns (group)
fig, axes = plt.subplots(3, 2, figsize=(14, 15), sharey='row')

panels = [
    ("Pairwise Correlation ($r_{sc}$)",  'r_sc',       a_rsc_99, a_val_rsc, b_rsc_99, b_val_rsc),
    ("Cosine Similarity",                'similarity', a_sim_99, a_val_sim, b_sim_99, b_val_sim),
    ("Pop Vector Corr (Pearson)",        'pop_corr',   a_pop_99, a_val_pop, b_pop_99, b_val_pop),
]

for row, (panel_title, key, a99, a_val, b99, b_val) in enumerate(panels):
    # Group A
    ax = axes[row, 0]
    ax.hist(group_a_chained_null[key], bins=40, color=NULL_COLOR, alpha=0.5, label='Chained Null')
    ax.axvline(a99,   color=THRESHOLD_COLOR, linestyle='--', label='99%')
    ax.axvline(a_val, color=GROUP_A_COLOR,   linewidth=3, label=f'Pooled Boot ({a_val:.4f})')
    ax.set_title(f'{GROUP_A_LABEL.upper()}: {panel_title}', fontsize=14)
    ax.legend(fontsize=8)
    if row == 2:
        ax.set_xlabel('Mean correlation / similarity', fontsize=12)

    # Group B
    ax = axes[row, 1]
    ax.hist(group_b_chained_null[key], bins=40, color=NULL_COLOR, alpha=0.5, label='Chained Null')
    ax.axvline(b99,   color=THRESHOLD_COLOR, linestyle='--', label='99%')
    ax.axvline(b_val, color=GROUP_B_COLOR,   linewidth=3, label=f'Pooled Boot ({b_val:.4f})')
    ax.set_title(f'{GROUP_B_LABEL.upper()}: {panel_title}', fontsize=14)
    ax.legend(fontsize=8)
    if row == 2:
        ax.set_xlabel('Mean correlation / similarity', fontsize=12)

# Cleanup spines
for ax_row in axes:
    for ax in ax_row:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

fig.suptitle(
    f"Coactivity by `{CATEGORY_COLUMN}`  ·  "
    f"{GROUP_A_LABEL} (IDs={GROUP_A_IDS}) vs {GROUP_B_LABEL} (IDs={GROUP_B_IDS})",
    fontsize=13,
)
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

## Per-session pop_corr — how the metric behaves session-by-session

In [ ]:
### Per-session pop_corr — how the metric behaves session-by-session

# For the chosen animal, plot one panel per session showing:
#   * histogram = within-session circular-shuffle null distribution of
#     pop_corr (combined across the two groups' onset templates so the
#     null reflects neural-data shifts, not category-specific timing).
#   * vertical line in GROUP_A_COLOR = observed pop_corr for group A
#     in that session.
#   * vertical line in GROUP_B_COLOR = observed pop_corr for group B
#     in that session.
#   * dashed line at the 99th percentile of the null.
#
# Sessions with too few trials in either group (<2) are skipped.



valid_session_rows = []
for sess_idx, sess in enumerate(sessions_data):
    a_starts = sess['group_a_df']['start'].to_numpy()
    b_starts = sess['group_b_df']['start'].to_numpy()
    if len(a_starts) < 2 or len(b_starts) < 2:
        continue
    a_mat = engine.extract_snippet_matrix(a_starts, sess['neural_data'], WINDOW_S)
    b_mat = engine.extract_snippet_matrix(b_starts, sess['neural_data'], WINDOW_S)
    pop_a = engine.compute_coactivity_metrics(a_mat)['pop_corr']
    pop_b = engine.compute_coactivity_metrics(b_mat)['pop_corr']
    # single-session circular shuffle null using BOTH groups' onsets so
    # the null sample size matches the joint trial count.
    joint_onsets = np.concatenate([a_starts, b_starts])
    null_distribution = engine.perform_circular_shuffle(
        onsets=joint_onsets,
        neural_data=sess['neural_data'],
        total_duration=sess['total_duration'],
        window_s=WINDOW_S,
        n_shuffles=PER_SESSION_N_SHUFFLES,
        seed=SEED + 100 + sess_idx,
    )
    valid_session_rows.append({
        'session_id': sess['session_id'],
        'n_a': len(a_starts), 'n_b': len(b_starts),
        'pop_a': pop_a, 'pop_b': pop_b,
        'null': null_distribution['pop_corr'],
    })

if not valid_session_rows:
    print("No sessions with sufficient trials in BOTH groups; figure skipped.")
else:
    n_panels = len(valid_session_rows)
    n_cols = min(4, n_panels)
    n_rows = int(np.ceil(n_panels / n_cols))
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(4.0 * n_cols, 3.0 * n_rows),
        sharex=False, sharey=False, squeeze=False,
    )
    for idx, row in enumerate(valid_session_rows):
        ax = axes[idx // n_cols, idx % n_cols]
        ax.hist(row['null'], bins=30, color=NULL_COLOR, alpha=0.55)
        thr99 = float(np.percentile(row['null'], 99))
        ax.axvline(thr99, color=THRESHOLD_COLOR, linestyle='--', linewidth=0.8)
        ax.axvline(
            row['pop_a'], color=GROUP_A_COLOR, linewidth=2.2,
            label=f"{GROUP_A_LABEL} (n={row['n_a']})  {row['pop_a']:.3f}",
        )
        ax.axvline(
            row['pop_b'], color=GROUP_B_COLOR, linewidth=2.2,
            label=f"{GROUP_B_LABEL} (n={row['n_b']})  {row['pop_b']:.3f}",
        )
        ax.set_title(row['session_id'], fontsize=9)
        ax.set_xlabel('pop_corr', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=7, loc='upper right')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    # blank any unused panels
    for idx in range(n_panels, n_rows * n_cols):
        axes[idx // n_cols, idx % n_cols].axis('off')

    fig.suptitle(
        f"Per-session pop_corr — {CHOSEN_ANIMAL}  ·  "
        f"{GROUP_A_LABEL} vs {GROUP_B_LABEL} ({CATEGORY_COLUMN})",
        fontsize=11,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()

## Cross-animal pop_corr summary

In [ ]:
### Cross-animal pop_corr summary

# Loops over every focal mouse in `ANIMALS_TO_SESSIONS`, loads its
# data (each animal -> its single best day), runs the pooled-bootstrap +
# label-permutation test (same flow as the compute cell above), and
# aggregates the per-animal pop_corr_A / pop_corr_B observed means plus
# the two-tailed permutation p for the A-vs-B difference.
#
# Slope plot: one line per animal connecting `pop_corr(group_A)` (left)
# to `pop_corr(group_B)` (right). Significant animals (p_two<0.05) are
# rendered in `GROUP_A_COLOR` if A > B, `GROUP_B_COLOR` if B > A; n.s.
# animals are gray. The aggregate mean line is overlaid in
# `THRESHOLD_COLOR`. Hyperparameters reuse the Parameters cell.

cross_animal_results: dict[str, dict] = {}
for animal_idx, (animal_id, session_names) in enumerate(ANIMALS_TO_SESSIONS.items()):
    print(f"Animal {animal_id} ({len(session_names)} sessions) ...", flush=True)
    animal_sessions = engine.load_animal_sessions(
        animal_id, session_names,
        data_root=DATA_ROOT, catalog=catalog,
        category_column=CATEGORY_COLUMN, group_a_ids=GROUP_A_IDS, group_b_ids=GROUP_B_IDS,
        cluster_group=UNIT_CLUSTER_GROUP, require_somatic=UNIT_REQUIRE_SOMATIC,
        brain_areas=UNIT_BRAIN_AREAS,
    )
    if not animal_sessions:
        print("  no sessions loaded, skipping")
        continue

    # Pool count matrices across the animal's sessions.
    a_mats, b_mats = [], []
    for sess in animal_sessions:
        a_starts = sess['group_a_df']['start'].to_numpy()
        b_starts = sess['group_b_df']['start'].to_numpy()
        if len(a_starts) < 1 or len(b_starts) < 1:
            continue
        a_mats.append(engine.extract_snippet_matrix(a_starts, sess['neural_data'], WINDOW_S))
        b_mats.append(engine.extract_snippet_matrix(b_starts, sess['neural_data'], WINDOW_S))
    if not a_mats or not b_mats:
        print("  insufficient trials, skipping")
        continue
    pooled_a = np.hstack(a_mats)
    pooled_b = np.hstack(b_mats)

    # Bootstrap-matched pop_corr means.
    bootstrap_target = min(USV_BOOTSTRAP_NUM, pooled_a.shape[1], pooled_b.shape[1])
    boot_a = engine.bootstrap_coactivity_distribution(pooled_a, bootstrap_target, N_BOOT_ITERATIONS, seed=SEED + 200 + 3 * animal_idx)
    boot_b = engine.bootstrap_coactivity_distribution(pooled_b, bootstrap_target, N_BOOT_ITERATIONS, seed=SEED + 200 + 3 * animal_idx + 1)
    pop_a_obs = float(np.mean(boot_a['pop_corr']))
    pop_b_obs = float(np.mean(boot_b['pop_corr']))

    # Label permutation test on the pooled trials (direct A vs B test).
    perm = engine.perform_label_permutation_test(
        counts_a=pooled_a, counts_b=pooled_b, n_permutations=N_PERMUTATIONS,
        seed=SEED + 200 + 3 * animal_idx + 2,
    )
    p_two = perm['pop_corr']['p_two_tailed']
    p_a_gt_b = perm['pop_corr']['p_a_gt_b']
    z_score = perm['pop_corr']['z_score']

    cross_animal_results[animal_id] = {
        'n_sessions':   len(animal_sessions),
        'n_a':          pooled_a.shape[1],
        'n_b':          pooled_b.shape[1],
        'n_units':      pooled_a.shape[0],
        'pop_a':        pop_a_obs,
        'pop_b':        pop_b_obs,
        'p_two':        p_two,
        'p_a_gt_b':     p_a_gt_b,
        'z':            z_score,
    }
    print(
        f"  units={pooled_a.shape[0]:>3}  n_a={pooled_a.shape[1]:>5}  n_b={pooled_b.shape[1]:>5}"
        f"  pop_a={pop_a_obs:+.4f}  pop_b={pop_b_obs:+.4f}"
        f"  Δ={pop_a_obs-pop_b_obs:+.4f}  p_two={p_two:.3f}  Z={z_score:+.2f}"
    )

# Slope plot
fig, ax = plt.subplots(figsize=(6.5, 5.0))
sig_alpha = 0.05
for animal_id, r in cross_animal_results.items():
    delta = r['pop_a'] - r['pop_b']
    if r['p_two'] < sig_alpha:
        line_color = GROUP_A_COLOR if delta > 0 else GROUP_B_COLOR
        line_alpha = 0.95
    else:
        line_color = NULL_COLOR
        line_alpha = 0.55
    ax.plot([0, 1], [r['pop_a'], r['pop_b']], color=line_color, alpha=line_alpha, linewidth=2.0, marker='o')
    ax.text(
        1.03, r['pop_b'],
        f"{animal_id}  (p={r['p_two']:.3f}{', *' if r['p_two'] < sig_alpha else ''})",
        fontsize=8, va='center', color=line_color,
    )

# Aggregate mean line
mean_a = float(np.mean([r['pop_a'] for r in cross_animal_results.values()]))
mean_b = float(np.mean([r['pop_b'] for r in cross_animal_results.values()]))
ax.plot([0, 1], [mean_a, mean_b], color=THRESHOLD_COLOR, linewidth=3.0, linestyle='--', label="mean of animals")
ax.set_xticks([0, 1])
ax.set_xticklabels([f"{GROUP_A_LABEL}\n(IDs={GROUP_A_IDS})", f"{GROUP_B_LABEL}\n(IDs={GROUP_B_IDS})"])
ax.set_xlim(-0.15, 1.6)
ax.set_ylabel('pop_corr (bootstrap mean)', fontsize=10)
ax.set_title(
    f"Cross-animal pop_corr  ·  {CATEGORY_COLUMN}  ·  "
    f"N={len(cross_animal_results)} mice",
    fontsize=11,
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Amplitude-stratified pop_corr

The confound check above shows the groups **differ** acoustically (complex calls are markedly louder). This section asks whether that difference actually **drives** the headline `pop_corr` effect, or is incidental. We bin all calls by their 30 ms RMS into quantile bins and, for bins holding at least `MIN_BIN_TRIALS` of **both** groups (the overlap region), bootstrap each group to a matched N and run the same `complex`-vs-`simple` label-permutation test as the main analysis — now *within* a loudness-matched bin.

Read it like this: if the `complex > simple` `pop_corr` gap **persists within bins** (where loudness is held roughly constant), loudness is **not** the explanation and the main result stands. If the gap **collapses within bins**, the effect was a loudness artifact and amplitude must be controlled directly. Because `pop_corr` is mean-centred (per-trial firing-rate magnitude removed) and PAG amplitude is plausibly a *downstream consequence* of the population command rather than an input to it, the gap surviving here is the expected — and reassuring — outcome.

In [ ]:
### Amplitude-stratified pop_corr: does the complex-vs-simple effect survive within amplitude bins?

# Stratification knobs.
N_AMPLITUDE_BINS = 5
MIN_BIN_TRIALS = 15   # required per group, per bin, for a bin to be compared

# Reuse the per-call RMS from the acoustic confound check above (aligned to the same
# session + dataframe order as the count matrices built here), then pool the spike-
# count matrices exactly as the main analysis does.
a_rms = group_a_acoustics['rms']
b_rms = group_b_acoustics['rms']
a_counts = np.hstack([
    engine.extract_snippet_matrix(sess['group_a_df']['start'].to_numpy(), sess['neural_data'], WINDOW_S)
    for sess in sessions_data
])
b_counts = np.hstack([
    engine.extract_snippet_matrix(sess['group_b_df']['start'].to_numpy(), sess['neural_data'], WINDOW_S)
    for sess in sessions_data
])
assert a_counts.shape[1] == a_rms.shape[0] and b_counts.shape[1] == b_rms.shape[0], (
    "RMS / count-matrix misalignment — re-run the acoustic confound check cell first."
)

# Quantile bin edges over the pooled finite-positive RMS of both groups.
pooled_rms = np.concatenate([a_rms, b_rms])
pooled_rms = pooled_rms[np.isfinite(pooled_rms) & (pooled_rms > 0)]
bin_edges = np.quantile(pooled_rms, np.linspace(0.0, 1.0, N_AMPLITUDE_BINS + 1))
bin_edges[-1] = np.nextafter(bin_edges[-1], np.inf)   # make the top edge inclusive

stratified_rows = []
for bin_idx in range(N_AMPLITUDE_BINS):
    lo, hi = bin_edges[bin_idx], bin_edges[bin_idx + 1]
    a_sel = np.isfinite(a_rms) & (a_rms >= lo) & (a_rms < hi)
    b_sel = np.isfinite(b_rms) & (b_rms >= lo) & (b_rms < hi)
    n_a, n_b = int(a_sel.sum()), int(b_sel.sum())
    row = {'lo': lo, 'hi': hi, 'n_a': n_a, 'n_b': n_b, 'pop_a': np.nan, 'pop_b': np.nan, 'p_two': np.nan}
    if n_a >= MIN_BIN_TRIALS and n_b >= MIN_BIN_TRIALS:
        a_bin, b_bin = a_counts[:, a_sel], b_counts[:, b_sel]
        n_match = min(n_a, n_b)   # matched N within the bin -> fair pop_corr comparison
        boot_a = engine.bootstrap_coactivity_distribution(a_bin, n_match, N_BOOT_ITERATIONS, seed=SEED + 300 + bin_idx)
        boot_b = engine.bootstrap_coactivity_distribution(b_bin, n_match, N_BOOT_ITERATIONS, seed=SEED + 400 + bin_idx)
        perm = engine.perform_label_permutation_test(
            counts_a=a_bin, counts_b=b_bin, n_permutations=N_PERMUTATIONS, seed=SEED + 500 + bin_idx,
        )
        row['pop_a'] = float(np.mean(boot_a['pop_corr']))
        row['pop_b'] = float(np.mean(boot_b['pop_corr']))
        row['p_two'] = perm['pop_corr']['p_two_tailed']
    stratified_rows.append(row)

# Unstratified reference (all trials, matched-N bootstrap) for context.
n_overall = min(a_counts.shape[1], b_counts.shape[1], USV_BOOTSTRAP_NUM)
pop_a_overall = float(np.mean(
    engine.bootstrap_coactivity_distribution(a_counts, n_overall, N_BOOT_ITERATIONS, seed=SEED + 600)['pop_corr']
))
pop_b_overall = float(np.mean(
    engine.bootstrap_coactivity_distribution(b_counts, n_overall, N_BOOT_ITERATIONS, seed=SEED + 601)['pop_corr']
))

print(f"Amplitude-stratified pop_corr — {CHOSEN_ANIMAL}  ({N_AMPLITUDE_BINS} RMS quantile bins)")
print(
    f"  unstratified: {GROUP_A_LABEL} pop_corr={pop_a_overall:+.4f}  "
    f"{GROUP_B_LABEL}={pop_b_overall:+.4f}  Δ={pop_a_overall - pop_b_overall:+.4f}\n"
)
print(f"  {'RMS bin':>24} {'n_a':>5} {'n_b':>5}  {'pop_a':>8} {'pop_b':>8} {'Δ':>8} {'p_two':>7}")
for r in stratified_rows:
    label = f"[{r['lo']:.2g}, {r['hi']:.2g})"
    if np.isnan(r['pop_a']):
        print(f"  {label:>24} {r['n_a']:>5} {r['n_b']:>5}   (too few trials in a group)")
    else:
        print(
            f"  {label:>24} {r['n_a']:>5} {r['n_b']:>5}  "
            f"{r['pop_a']:>+8.4f} {r['pop_b']:>+8.4f} {r['pop_a'] - r['pop_b']:>+8.4f} {r['p_two']:>7.4f}"
        )

# Plot pop_corr per amplitude bin (complex vs simple), with the unstratified means as references.
valid = [r for r in stratified_rows if not np.isnan(r['pop_a'])]
fig, ax = plt.subplots(figsize=(7.0, 4.5))
if valid:
    centers = [float(np.sqrt(r['lo'] * r['hi'])) for r in valid]
    ax.plot(centers, [r['pop_a'] for r in valid], '-o', color=GROUP_A_COLOR, label=GROUP_A_LABEL)
    ax.plot(centers, [r['pop_b'] for r in valid], '-o', color=GROUP_B_COLOR, label=GROUP_B_LABEL)
    for r, c in zip(valid, centers):
        if r['p_two'] < 0.05:
            ax.annotate('*', (c, max(r['pop_a'], r['pop_b'])), ha='center', va='bottom', fontsize=14, color=THRESHOLD_COLOR)
    ax.set_xscale('log')
    ax.axhline(pop_a_overall, color=GROUP_A_COLOR, linestyle='--', linewidth=1, alpha=0.6)
    ax.axhline(pop_b_overall, color=GROUP_B_COLOR, linestyle='--', linewidth=1, alpha=0.6)
else:
    ax.text(0.5, 0.5, "No amplitude bin held enough trials in both groups", ha='center', transform=ax.transAxes)
ax.set_xlabel('30 ms RMS amplitude (bin geometric centre, log)', fontsize=10)
ax.set_ylabel('pop_corr (bootstrap mean, matched N per bin)', fontsize=10)
ax.set_title(
    f"Amplitude-stratified pop_corr — {CHOSEN_ANIMAL}\n"
    f"dashed = unstratified means; * = within-bin {GROUP_A_LABEL}-vs-{GROUP_B_LABEL} p<0.05",
    fontsize=10,
)
ax.legend(frameon=False, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
plt.show()